<a href="https://colab.research.google.com/github/iam4tart/speech-lab/blob/main/01-glowtts-from-scratch/inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install coqui-tts

In [ ]:
!pip uninstall -y transformers
!pip install transformers==4.57.5

In [16]:
from TTS.api import TTS

In [17]:
import torch
import soundfile as sf

from TTS.config import load_config
from TTS.tts.models.glow_tts import GlowTTS
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from TTS.utils.audio import AudioProcessor

In [18]:
CONFIG_PATH = "config.json"
MODEL_PATH = "best_model.pth"
OUT_PATH = "output.wav"

In [19]:
config = load_config(CONFIG_PATH)

model = GlowTTS(config)
ckpt = torch.load(MODEL_PATH, map_location="cpu")

if "model" in ckpt:
    model.load_state_dict(ckpt["model"])
else:
    model.load_state_dict(ckpt)

model.eval()

GlowTTS(
  (encoder): Encoder(
    (emb): Embedding(67, 192)
    (prenet): ResidualConv1dLayerNormBlock(
      (conv_layers): ModuleList(
        (0-2): 3 x Conv1d(192, 192, kernel_size=(5,), stride=(1,), padding=(2,))
      )
      (norm_layers): ModuleList(
        (0-2): 3 x LayerNorm()
      )
      (proj): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
    )
    (encoder): RelativePositionTransformer(
      (dropout): Dropout(p=0.1, inplace=False)
      (attn_layers): ModuleList(
        (0-5): 6 x RelativePositionMultiHeadAttention(
          (conv_q): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_k): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_v): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (conv_o): Conv1d(192, 192, kernel_size=(1,), stride=(1,))
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
      (norm_layers_1): ModuleList(
        (0-5): 6 x LayerNorm()
      )
      (ffn_layers): ModuleList(
       

In [20]:
tokenizer, config = TTSTokenizer.init_from_config(config)

In [21]:
text = "have you played that new game called animal well which has no combat but amazing visuals"

tokens = tokenizer.text_to_ids(text)
tokens = torch.LongTensor(tokens).unsqueeze(0)

In [22]:
x_lengths = torch.LongTensor([tokens.shape[1]])
with torch.no_grad():
    out = model.inference(tokens, aux_input={"x_lengths": x_lengths})
mel = out["model_outputs"]

In [23]:
ap = AudioProcessor(**config.audio)

# Griffin-Lim vocoder
wav = ap.inv_melspectrogram(mel[0].cpu().numpy().T)

In [24]:
sf.write(OUT_PATH, wav, config.audio["sample_rate"])
print("saved:", OUT_PATH)

saved: output.wav


In [25]:
from IPython.display import Audio
Audio(OUT_PATH)